# <b>Second Stage: Data Preparation</b>

# 2. TCA Cycle Enzymes and Flux Combinations

## 2.1 General Workflow

<div style='text-align: justify'>The following cells present the code used to identify the main TCA cycle enzymes responsible for NADH production under anaerobic conditions. The objective is to determine which enzymes increase NADH production without causing a substantial reduction in biomass formation. To achieve this, we applied machine learning to evaluate the feature importance of all TCA cycle enzymes across two specific phenotypes:</div>
<ol>
    <li>NADH production above X <b>and</b> biomass production above y;</li>
    <li>NADH production below X <b>or</b> biomass production below y.</li>
</ol>

<div style='text-align: justify'>First, we selected all enzymes associated with the TCA cycle in the iCH360 model and performed Flux Variability Analysis (FVA) to identify reactions blocked under anaerobic conditions and to determine the permissible flux ranges that still allow optimal biomass production. Next, for each enzyme, we generated 10 flux values within the interval established by FVA. Using these values, we created 1.000.000 combinations of enzyme flux distributions and performed Flux Balance Analysis (FBA) for each combination, with biomass as the objective function. In addition to biomass formation, NADH production was calculated for every combination. The results were compiled into a CSV file in which each row corresponds to a flux combination, including the associated biomass and NADH production values. The columns represent the selected enzymes, NADH production, and biomass formation. Finally, a Random Forest model was applied to determine the relative importance of each enzyme with respect to the defined classes.</div><br>

General Workflow:
<ol>
    <li>Select TCA cycle enzymes</li>
    <li>Perform FVA for each enzyme under anaerobic conditions, using biomass as the objective function</li>
    <li>Generate 10 flux values for each TCA cycle enzyme within the range established by FVA</li>
    <li>Create 1.000.000 distinct combinations of flux distributions for the selected TCA cycle enzymes</li>
    <li>Constrain enzyme bounds according to each flux combination and perform FBA</li>
    <li>Calculate biomass and NADH production for each combination</li>
    <li>Define the groups used to classify biomass and NADH production levels</li>
    <li>Apply a Random Forest model to determine the relative importance of each TCA cycle enzyme in class discrimination</li>
</ol>

## 2.2 Selection of TCA Cycle Enzymes and Generation of Flux Combinations

<div style='text-align: justify'>The initially selected enzymes comprised the core TCA cycle enzymes in the EC_iCH360 model. To further assess the role of enzymes that mediate entry into the cycle, pyruvate dehydrogenase (PDH) and phosphoenolpyruvate carboxylase (PPC) were also included in the analysis.</div><br>
<div style='text-align: justify'><b>TCA cycle enzymes:</b></div>
<ol>
    <li><b>Citrate synthase (CS):</b> $acetil$-$CoA + H_2O + oxaloacetate$ $\rightarrow$ $citrate + CoA + H^+$ </li>
    <li><b>Citrate hydro-lyase (ACONTa) (Aconitase [half-reation A]):</b> $citrate$ &harr; $cis$-$aconitate + H_2O$</li>
    <li><b>Isocitrate hydro-lyase (ACONTb) (Aconitase [half-reation B]):</b> $cis$-$aconitate + H_2O$ &harr; $isocitrate$</li>
    <li><b>Isocitrate dehydrogenase (ICDHyr): $isocitrate + NADP^+$ &harr; $2$-$oxoglutarate + CO_2 + NADPH$ </b></li>
    <li><b>Isocitrate lyase (ICL): $isocitrate$ &harr; $glyoxylate + succinate$</b></li>
    <li><b>Malate synthase(MALS):</b> $glyoxylate + H_2O$ $\rightarrow$ $CoA + H^+ + L$-$malate$ </li>
    <li><b>2-Oxoglutarate dehydrogenase (AKGDH):</b> $2$-$oxoglutarate + CoA + NAD^+$ $\rightarrow$ $CO_2 + NADH + succinyl$-$CoA$</li>
    <li><b>Succinyl-CoA synthetase (SUCOAS):</b> $ADP + P_i + succinyl$-$CoA$ &harr; $ATP + CoA + succinate$</li>
    <li><b>Succinate dehydrogenase (SUCDi):</b> $ubiquinone$-$8$ $+$ $succinate$ $\rightarrow$ $fumarate + ubiquinol$-$8$</li>
    <li><b>Fumarate reductase (FRD2):</b> $fumarate + menaquinol$-$8$ $\rightarrow$ $menaquinone$-$8$ $+$ $succinate$</li>
    <li><b>Fumarase (FUM):</b> $fumarate + H_2O$ &harr; $L$-$malate$</li>
</ol>
<div style='text-align: justify'><b>Entry enzymes:</b></div>
<ol>
    <li><b>Pyruvate dehydrogenase (PDH):</b> $pyruvate + NAD^+ + CoA$ $\rightarrow$ $acetyl$-$CoA + CO_2 + NADH$</li>
    <li><b>Phosphoenolpyruvate carboxylase (PPC):</b> $CO_2 + phosphoenolpyruvate + H_2O$ $\rightarrow$ $oxaloacetate + H^+ + P_i$</li>
</ol>

<div style='text-align: justify'><b>Note 1:</b> For reversible reactions, the EC_iCH360 model separates each enzyme into two stoichiometric reactions, identified by the suffixes “_fw” (forward) and “_bw” (backward) in the reaction ID (<a href="https://doi.org/10.1371/journal.pcbi.1013564">Corrao et al., 2025</a>). This structure provides insight into whether flux preferentially proceeds through the oxidative or reductive branch following specific modifications.</div><br>

<div style='text-align: justify'><b>Note 2:</b> Malate dehydrogenase (MDH) was not included in the analysis. Because the MDH reaction is reversible, flux modifications can artificially drive high NADH yields. In the EC_iCH360 model, large fluxes through “MDH_fw” are permitted when “MDH_bw” is constrained to operate at the same magnitude, generating an internal loop of NADH production. One alternative considered was to knock out “MDH_bw” and retain only “MDH_fw.” However, after evaluation, we opted to exclude MDH from the machine learning model. Consequently, MDH flux is expected to adjust in response to the remaining enzymes included in the analysis.</div>

In [1]:
# model handling and loading
import cobra
from cobra.io import load_json_model
from cobra.flux_analysis import flux_variability_analysis

# data manipulation and visualization
import pandas as pd
import numpy as np

In [3]:
# model loading
# model = load_json_model('../Models/EC_iCH360.json')

In [4]:
model

Name,EC_iCH360
Memory address,7f2de7dd7490
Number of metabolites,305
Number of reactions,505
Number of genes,356
Number of groups,0
Objective expression,1.0*Biomass_fw - 1.0*Biomass_fw_reverse_bf19d
Compartments,"cytosol, extracellular space, periplasm"


In [3]:
model_anaero = model.copy()

Read LP format model from file /tmp/tmpy2ons51v.lp
Reading time = 0.00 seconds
: 305 rows, 1010 columns, 4816 nonzeros


In [4]:
# Simulating an anaerobic environment
model_anaero.reactions.get_by_id('EX_o2_e_bw').bounds = (0,0)

In [5]:
# This list facilitates the future manipulation of the selected enzymes
R_imp = ['PDH_fw','PPC_fw','CS_fw','ACONTa_fw','ACONTa_bw',
         'ACONTb_fw','ACONTb_bw','ICDHyr_fw','ICDHyr_bw',
         'AKGDH_fw','SUCOAS_fw','SUCOAS_bw','FUM_fw','FUM_bw',
         'FRD2_fw','ICL_fw','ICL_bw','SUCDi_fw','MALS_fw']

In [6]:
# Here, FVA is performed to identify reactions that are blocked under anaerobic conditions.
# From this analysis, we also establish the permissible flux range for each TCA cycle enzyme.

loop_reactions = [model_anaero.reactions.get_by_id(i) for i in R_imp]
df_fva_anaero = flux_variability_analysis(model_anaero, reaction_list=loop_reactions,
                                   loopless=True,fraction_of_optimum=0.9)

display(df_fva_anaero)

,minimum,maximum
PDH_fw,0.000000,1.563670
PPC_fw,0.305451,3.604094
CS_fw,0.099471,0.490815
ACONTa_fw,0.099471,3.557768
ACONTa_bw,0.000000,3.458297
ACONTb_fw,0.099471,4.978967
ACONTb_bw,0.000000,4.879497
ICDHyr_fw,0.099471,2.648173
ICDHyr_bw,0.000000,2.548702
AKGDH_fw,0.000000,0.276540


<div style='text-align: justify'>As shown in the generated data frame, SUCDi is blocked under anaerobic conditions. Therefore, similar to MDH, it will be excluded from further analysis.</div>

In [7]:
# Final data frame
df_fva_anaero_final = df_fva_anaero.drop(["SUCDi_fw"],axis=0)
display(df_fva_anaero_final)

# Final set of reactions
R_imp_final = ['PDH_fw','PPC_fw','CS_fw','ACONTa_fw','ACONTa_bw',
               'ACONTb_fw','ACONTb_bw','ICDHyr_fw','ICDHyr_bw',
               'AKGDH_fw','SUCOAS_fw','SUCOAS_bw','FUM_fw','FUM_bw',
               'FRD2_fw','ICL_fw','ICL_bw','MALS_fw']

,minimum,maximum
PDH_fw,0.000000,1.563670
PPC_fw,0.305451,3.604094
CS_fw,0.099471,0.490815
ACONTa_fw,0.099471,3.557768
ACONTa_bw,0.000000,3.458297
ACONTb_fw,0.099471,4.978967
ACONTb_bw,0.000000,4.879497
ICDHyr_fw,0.099471,2.648173
ICDHyr_bw,0.000000,2.548702
AKGDH_fw,0.000000,0.276540


## 2.3 Generation of Flux Combinations among TCA Cycle Enzymes

In [8]:
# Creation of a dictionary containing all possible flux values for each enzyme.

"""We selected 10 flux values between the FVA minimum and maximum bounds to simplify the generation of combinations and reduce the 
computational cost of the simulations."""
 
dataset = {i:list(np.linspace(df_fva_anaero_final.loc[i,'minimum'],df_fva_anaero_final.loc[i,'maximum'],10)) 
           for i in df_fva_anaero_final.index}

In [9]:
# For better visualization of the flux values
df_dataset = pd.DataFrame(dataset)
df_dataset = df_dataset.transpose()
df_dataset.columns = ['1','2','3','4','5','6','7','8','9','10']
display(df_dataset)

,1,2,3,4,5,6,7,8,9,10
PDH_fw,0.000000,0.173741,0.347482,0.521223,0.694965,0.868706,1.042447,1.216188,1.389929,1.563670
PPC_fw,0.305451,0.671967,1.038482,1.404998,1.771514,2.138030,2.504546,2.871062,3.237578,3.604094
CS_fw,0.099471,0.142954,0.186436,0.229919,0.273401,0.316884,0.360367,0.403849,0.447332,0.490815
ACONTa_fw,0.099471,0.483726,0.867981,1.252237,1.636492,2.020747,2.405002,2.789257,3.173513,3.557768
ACONTa_bw,0.000000,0.384255,0.768510,1.152766,1.537021,1.921276,2.305531,2.689786,3.074042,3.458297
ACONTb_fw,0.099471,0.641637,1.183804,1.725970,2.268136,2.810302,3.352469,3.894635,4.436801,4.978967
ACONTb_bw,0.000000,0.542166,1.084333,1.626499,2.168665,2.710831,3.252998,3.795164,4.337330,4.879497
ICDHyr_fw,0.099471,0.382660,0.665849,0.949038,1.232227,1.515416,1.798605,2.081794,2.364984,2.648173
ICDHyr_bw,0.000000,0.283189,0.566378,0.849567,1.132756,1.415945,1.699134,1.982324,2.265513,2.548702
AKGDH_fw,0.000000,0.030727,0.061453,0.092180,0.122907,0.153634,0.184360,0.215087,0.245814,0.276540


In [10]:
# Packages required to generate 1.000.000 random combinations of flux values
from iteration_utilities import random_product

# To ensure reproducibility of subsequent simulations.
import random
random.seed(42)

def random_combinations(size,data,reactions):

    # Flux values to be used for generating the combinations
    values = list(data.values())
    combinations = []
    
    # For the initial combination (no duplication check required)
    y = True
    
    for i in range(size):
        # a it's a combination 
        a = random_product(*values)
        
        # If y, so it's unique
        if y:
            combinations.append(a)
            
            """After adding this combination to the set of combinations, we set y = False to indicate that subsequent combinations must 
            be checked for uniqueness before being accepted."""
            
            y = False
            continue

        # If y is False, we must verify whether the generated combination has already been added to the list.
        else:
            
            # If a is already in combinations, generate a new combination until one is found that is not yet included.
            if a in combinations:
                while True:
                    a = random_product(*values)
                    if a not in combinations:
                        combinations.append(a)
                        break

            # If a is not in combinations, append it directly to the list.
            else:
                combinations.append(a)
            
    df = pd.DataFrame(combinations)
    df.columns = reactions
    
    return df

In [ ]:
# The approximate time required to generate the combinations is about 4 hours (my spces: ryzen 7 5700U; 16 GB RAM).
df_combinations = random_combinations(1000000,dataset,R_imp_final)
display(df_combinations)

In [15]:
# The combinations must be saved for subsequent analysis. Remove the # to enable saving.

# path = '../Datasets/combinations.csv'
# df_combinations.to_csv(path,index=False)

In [12]:
df_combinations = pd.read_csv('../Datasets/combinations.csv')
display(df_combinations)

,PDH_fw,PPC_fw,CS_fw,ACONTa_fw,ACONTa_bw,ACONTb_fw,ACONTb_bw,ICDHyr_fw,ICDHyr_bw,AKGDH_fw,SUCOAS_fw,SUCOAS_bw,FUM_fw,FUM_bw,FRD2_fw,ICL_fw,ICL_bw,MALS_fw
0,0.173741,0.305451,0.273401,1.252237,1.152766,1.183804,0.542166,2.364984,0.283189,0.276540,2.957539,0.000000,0.000000,0.369804,0.656696,1.161967,3.098579,0.261842
1,0.000000,3.237578,0.229919,3.173513,2.305531,1.725970,3.795164,2.648173,1.132756,0.000000,0.985846,2.925053,1.852641,1.479215,0.448039,1.161967,1.936612,0.029094
2,0.173741,2.504546,0.142954,2.020747,1.921276,4.978967,2.168665,0.099471,1.982324,0.245814,0.492923,2.925053,0.370528,2.958430,0.865354,3.485901,1.936612,0.261842
3,0.521223,0.671967,0.099471,1.252237,1.537021,0.641637,1.626499,0.382660,1.699134,0.122907,3.450462,2.437544,0.741057,1.849019,1.074011,1.161967,1.549290,0.029094
4,1.563670,1.038482,0.447332,1.252237,0.768510,3.894635,3.252998,1.232227,2.265513,0.092180,2.464616,0.000000,1.111585,0.000000,1.074011,2.323934,1.549290,0.029094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,0.521223,0.305451,0.360367,1.636492,2.305531,3.894635,0.000000,1.515416,1.699134,0.000000,1.971692,3.412562,1.111585,2.958430,0.656696,3.485901,2.711257,0.058187
999996,1.563670,2.871062,0.360367,0.867981,1.537021,1.183804,0.000000,0.665849,0.566378,0.276540,3.943385,4.387580,0.000000,1.109411,0.030724,1.936612,1.161967,0.261842
999997,0.521223,0.305451,0.360367,2.789257,3.458297,2.268136,1.084333,2.648173,2.265513,0.000000,0.000000,3.412562,2.223170,1.109411,1.908640,3.098579,1.936612,0.145468
999998,1.216188,0.305451,0.403849,0.867981,0.768510,4.436801,4.879497,0.382660,0.849567,0.215087,3.943385,3.412562,0.000000,3.328234,0.865354,3.485901,1.549290,0.145468


<div style='text-align: justify'>As shown in the structure of the final data frame, each row of <b>combinations.csv</b> corresponds to a specific set of flux values for the selected enzymes, listed in the columns. In the next step, biomass and NADH production will be calculated for each combination under anaerobic conditions by performing 1,000,000 Flux Balance Analyses (FBA). For each case, enzyme flux bounds will be fixed according to the values defined in the respective combination. Infeasible solutions will be handled using the cnapy algorithm (see Part 3 for details).</div>

# End of Part 2!